# Prior Model — Deployment

Trains the prior XGBoost model on the **last 6 seasons** (2019-20 through 2025-26) using the hyperparameters and features tuned in `prior_modeling.ipynb`. No validation or test split — this is the production model.

In [2]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
from xgboost import XGBClassifier

DATA_DIR = Path('data')
MODEL_DIR = Path('prior_models')
MODEL_DIR.mkdir(exist_ok=True)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## 1. Load tuned config

In [7]:
# Load hyperparameters and feature list from the tuned model
config = json.load(open(MODEL_DIR / 'config.json'))
FEATURE_COLS = config['feature_cols']
BEST_ITERATION = config['best_iteration']
HYPERPARAMS = config['hyperparameters']

print(f'Features: {len(FEATURE_COLS)}')
print(f'Best iteration from tuning: {BEST_ITERATION}')
print(f'Hyperparameters: {HYPERPARAMS}')

Features: 159
Best iteration from tuning: 612
Hyperparameters: {'n_estimators': 1000, 'learning_rate': 0.02, 'max_depth': 2, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 8.0, 'min_child_weight': 15, 'early_stopping_rounds': 30}


## 2. Load & filter training data (last 6 seasons)

In [5]:
df = pd.read_csv(DATA_DIR / 'training_games4.csv')
df['game_date'] = pd.to_datetime(df['game_date'])

# Assign season start year: games before July belong to previous year's season
df['season'] = df['game_date'].apply(lambda d: d.year if d.month >= 7 else d.year - 1)

print(f'Total games: {len(df)}')
print(f'Seasons available: {sorted(df["season"].unique())}')
print(f'Date range: {df["game_date"].min().date()} to {df["game_date"].max().date()}')

Total games: 19562
Seasons available: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Date range: 2008-11-15 to 2026-04-04


In [9]:
# Last 6 seasons: 2019-20 through 2025-26
TRAIN_SEASONS = list(range(2020, 2026))
df_train = df[df['season'].isin(TRAIN_SEASONS)].copy()

print(f'Training on seasons: {TRAIN_SEASONS}')
print(f'Training games: {len(df_train)}')
print(f'Date range: {df_train["game_date"].min().date()} to {df_train["game_date"].max().date()}')
print(f'\nGames per season:')
print(df_train.groupby('season').size())

Training on seasons: [2020, 2021, 2022, 2023, 2024, 2025]
Training games: 6686
Date range: 2020-07-30 to 2026-04-04

Games per season:
season
2020    1128
2021    1174
2022    1163
2023    1163
2024    1051
2025    1007
dtype: int64


In [10]:
X_train = df_train[FEATURE_COLS]
y_train = (df_train['score_diff'] > 0).astype(int)

print(f'Feature matrix: {X_train.shape}')
print(f'Home win rate: {y_train.mean():.3f}')
print(f'Missing values: {X_train.isna().sum().sum()}')

Feature matrix: (6686, 159)
Home win rate: 0.552
Missing values: 0


## 3. Train deployment model

In [6]:
# Use the exact hyperparameters from tuning, with n_estimators set to the
# best iteration found during early stopping (no val set to stop on here)
model = XGBClassifier(
    n_estimators=BEST_ITERATION,
    learning_rate=HYPERPARAMS['learning_rate'],
    max_depth=HYPERPARAMS['max_depth'],
    subsample=HYPERPARAMS['subsample'],
    colsample_bytree=HYPERPARAMS['colsample_bytree'],
    reg_lambda=HYPERPARAMS['reg_lambda'],
    min_child_weight=HYPERPARAMS['min_child_weight'],
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print(f'Model trained: {model.n_estimators} trees')

Model trained: 612 trees


## 4. Sanity check on training data

In [7]:
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

y_proba = model.predict_proba(X_train)[:, 1]
y_pred = (y_proba > 0.5).astype(int)

print('Training set metrics (sanity check — NOT generalization):')
print(f'  Accuracy:    {accuracy_score(y_train, y_pred):.4f}')
print(f'  ROC-AUC:     {roc_auc_score(y_train, y_proba):.4f}')
print(f'  Brier Score: {brier_score_loss(y_train, y_proba):.4f}')
print(f'  Log Loss:    {log_loss(y_train, y_proba):.4f}')

Training set metrics (sanity check — NOT generalization):
  Accuracy:    0.6949
  ROC-AUC:     0.7672
  Brier Score: 0.1966
  Log Loss:    0.5770


## 5. Save deployment model

In [8]:
# Save model
deploy_path = MODEL_DIR / 'xgboost_prior_deploy.pkl'
with open(deploy_path, 'wb') as f:
    pickle.dump(model, f)
print(f'Saved deployment model to {deploy_path}')

# Save config
deploy_config = {
    'model_type': 'XGBClassifier',
    'n_estimators': model.n_estimators,
    'hyperparameters': HYPERPARAMS,
    'feature_cols': FEATURE_COLS,
    'train_seasons': TRAIN_SEASONS,
    'train_games': len(df_train),
    'train_date_range': [str(df_train['game_date'].min().date()), str(df_train['game_date'].max().date())],
}

with open(MODEL_DIR / 'config_deploy.json', 'w') as f:
    json.dump(deploy_config, f, indent=2)
print(f'Saved deployment config to {MODEL_DIR / "config_deploy.json"}')

Saved deployment model to prior_models/xgboost_prior_deploy.pkl
Saved deployment config to prior_models/config_deploy.json


## 6. Generate predictions for all games

In [12]:
# take our stored model and use it to predict on all games from 2010 onward
with open(MODEL_DIR / 'xgboost_prior_deploy.pkl', 'rb') as f:
    model = pickle.load(f)

deploy_config = json.load(open(MODEL_DIR / 'config_deploy.json'))
FEATURE_COLS = deploy_config['feature_cols']

df_all = df[df['season'] >= 2010].copy()
X_all = df_all[FEATURE_COLS]
all_proba = model.predict_proba(X_all)[:, 1]

predictions_df = pd.DataFrame({
    'game_id': df_all['game_id'].values,
    'game_date': df_all['game_date'].values,
    'home_team': df_all['home_team'].values,
    'away_team': df_all['away_team'].values,
    'prior_home_wp': np.round(all_proba, 6),
    'home_win_actual': ((df_all['score_diff'] > 0).astype(int)).values,
    'score_diff': df_all['score_diff'].values if 'score_diff' in df_all.columns else 0,
})

predictions_df.to_csv(DATA_DIR / 'games_predictions_deploy.csv', index=False)
print(f'Saved predictions to data/games_predictions_deploy.csv ({len(predictions_df):,} games)')
print(f'  Seasons: {sorted(df_all["season"].unique())}')
print(f'  Date range: {df_all["game_date"].min().date()} to {df_all["game_date"].max().date()}')
print(predictions_df.tail(10).to_string(index=False))

Saved predictions to data/games_predictions_deploy.csv (17,258 games)
  Seasons: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Date range: 2010-11-14 to 2026-04-04
  game_id  game_date home_team away_team  prior_home_wp  home_win_actual  score_diff
401810975 2026-04-03       CHA       IND       0.799156                1          21
401810978 2026-04-03        NY       CHI       0.815266                1          40
401810977 2026-04-03       BKN       ATL       0.150416                0         -34
401810981 2026-04-03       MIL       BOS       0.107471                0         -32
401810980 2026-04-03       MEM       TOR       0.273076                0         -32
401810979 2026-04-03       HOU      UTAH       0.893669                1          34
401810983 2026-04-03       SAC        NO       0.505851                1           4
401810984 2026-04-04       MIA       WSH       0.873324                1          16
401810985 2026-04-

In [13]:
# Predict on all games from 2010 onward (for downstream posterior pipeline)
df_all = df[df['season'] >= 2010].copy()
X_all = df_all[FEATURE_COLS]
all_proba = model.predict_proba(X_all)[:, 1]

predictions_df = pd.DataFrame({
    'game_id': df_all['game_id'].values,
    'game_date': df_all['game_date'].values,
    'home_team': df_all['home_team'].values,
    'away_team': df_all['away_team'].values,
    'prior_home_wp': np.round(all_proba, 6),
    'home_win_actual': ((df_all['score_diff'] > 0).astype(int)).values,
    'score_diff': df_all['score_diff'].values if 'score_diff' in df_all.columns else 0,
})

predictions_df.to_csv(DATA_DIR / 'games_predictions_deploy.csv', index=False)
print(f'Saved predictions to data/games_predictions_deploy.csv ({len(predictions_df):,} games)')
print(f'  Seasons: {sorted(df_all["season"].unique())}')
print(f'  Date range: {df_all["game_date"].min().date()} to {df_all["game_date"].max().date()}')
print(predictions_df.tail(10).to_string(index=False))

Saved predictions to data/games_predictions_deploy.csv (17,258 games)
  Seasons: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Date range: 2010-11-14 to 2026-04-04
  game_id  game_date home_team away_team  prior_home_wp  home_win_actual  score_diff
401810975 2026-04-03       CHA       IND       0.799156                1          21
401810978 2026-04-03        NY       CHI       0.815266                1          40
401810977 2026-04-03       BKN       ATL       0.150416                0         -34
401810981 2026-04-03       MIL       BOS       0.107471                0         -32
401810980 2026-04-03       MEM       TOR       0.273076                0         -32
401810979 2026-04-03       HOU      UTAH       0.893669                1          34
401810983 2026-04-03       SAC        NO       0.505851                1           4
401810984 2026-04-04       MIA       WSH       0.873324                1          16
401810985 2026-04-